In [2]:
!pip install chess
!pip install python-chess torch numpy matplotlib imageio

import chess
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 68.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=8b14e7e36cfe1bddc1a29312b628b60f6601161e19297ec9e60f8f2d1eda9c1c
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


## State Representation

In [3]:

def board_to_tensor(board):
    piece_map = board.piece_map()
    state = np.zeros((12, 8, 8), dtype=np.float32)

    for square, piece in piece_map.items():
        row = square // 8
        col = square % 8
        idx = piece.piece_type - 1 + (0 if piece.color else 6)
        state[idx, row, col] = 1

    return state.flatten()


## DQN Model

In [4]:

class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, action_size)
        )

    def forward(self, x):
        return self.net(x)


## Agent

In [5]:

class Agent:
    def __init__(self):
        self.state_size = 12*8*8
        self.action_size = 64*64

        self.model = DQN(self.state_size, self.action_size)
        self.optimizer = optim.Adam(self.model.parameters(), lr=1e-3)
        self.memory = deque(maxlen=5000)

        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.1

    def act(self, state, legal_moves):
        if random.random() < self.epsilon:
            return random.choice(list(legal_moves))

        state_t = torch.tensor(state).float().unsqueeze(0)
        q_values = self.model(state_t).detach().numpy()[0]

        best_move = None
        best_q = -1e9

        for move in legal_moves:
            idx = move.from_square * 64 + move.to_square
            if q_values[idx] > best_q:
                best_q = q_values[idx]
                best_move = move

        return best_move

    def remember(self, *args):
        self.memory.append(args)

    def replay(self, batch_size=32):
        if len(self.memory) < batch_size:
            return

        batch = random.sample(self.memory, batch_size)

        for state, action_idx, reward, next_state, done in batch:
            target = reward
            if not done:
                target += self.gamma * torch.max(self.model(torch.tensor(next_state).float())).item()

            target_f = self.model(torch.tensor(state).float())
            target_f[action_idx] = target

            loss = nn.MSELoss()(self.model(torch.tensor(state).float()), target_f)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay


## Training Loop

In [ ]:

agent = Agent()
episodes = 50

results = {"win":0, "loss":0, "draw":0}

for ep in range(episodes):
    board = chess.Board()

    while not board.is_game_over():
        state = board_to_tensor(board)
        move = agent.act(state, board.legal_moves)

        action_idx = move.from_square * 64 + move.to_square
        board.push(move)

        if board.is_game_over():
            result = board.result()
            if result == "1-0":
                reward = 1
                results["win"] += 1
            elif result == "0-1":
                reward = -1
                results["loss"] += 1
            else:
                reward = 0
                results["draw"] += 1

            next_state = board_to_tensor(board)
            agent.remember(state, action_idx, reward, next_state, True)
            break

        # random opponent
        opp_move = random.choice(list(board.legal_moves))
        board.push(opp_move)

        next_state = board_to_tensor(board)
        agent.remember(state, action_idx, 0, next_state, False)
        agent.replay()

    print(f"Episode {ep+1} done")

print("Results:", results)


Episode 1 done
Episode 2 done


## Save Model

In [ ]:

torch.save(agent.model.state_dict(), "chess_dqn.pth")
